# Adversarial & Property-based Validity Tests

**Goal:** Test the new reuse validity checker against an independently implemented event-level reference scheduler on the 50 saved Experiment 2 plans and controlled structural mutations. **Not** the algorithmic Experiment 5; no GidNET reruns, SABRE routing, or quantum simulation.

You must place `reuse_adversarial.py`, `run_validity_adversarial.py` and `reuse_validity.py` in your project root. Recommended: run `python -m pytest -q test_reuse_validity.py test_reuse_validity_adversarial.py` first.

**Interpretation:** Both checkers address static dependency feasibility. Agreement is *not* an independent proof of preserved measurement/reset semantics.

In [1]:
from pathlib import Path
from types import SimpleNamespace
from run_validity_adversarial import run

# Set this path exactly as in validate_existing_plans.ipynb
BENCH_DIR = Path(r"C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks")
PLAN_CSV = Path("experiment2_results/experiment2_plan_features.csv")
PREVIOUS_VALIDATION = Path("validity_checker_results/validation_of_experiment2_saved_plans.csv")
OUT_DIR = Path("validity_adversarial_results")
PER_OPERATOR = 12      # set to 30 or higher only after the first smoke run
SEED = 20261011

assert BENCH_DIR.is_dir(), f"Incorrect BENCH_DIR: {BENCH_DIR}"
assert PLAN_CSV.is_file(), f"Missing: {PLAN_CSV}"
assert PREVIOUS_VALIDATION.is_file(), "Run validate_existing_plans.ipynb first for complete width audit"
args = SimpleNamespace(bench_dir=BENCH_DIR, plan_csv=PLAN_CSV,
    validation_csv=PREVIOUS_VALIDATION, out_dir=OUT_DIR,
    per_operator=PER_OPERATOR, seed=SEED)
print("Configuration OK")

Configuration OK


In [2]:
status = run(args)
assert status == 0, "Validation audit found disagreement/failures. Inspect output files before moving on."

Checking alu-bdd_288.qasm: 6 saved plans
Checking mini_alu_305.qasm: 16 saved plans
Checking rd73_140.qasm: 6 saved plans
Checking sym9_146.qasm: 9 saved plans
Checking system9.qasm: 13 saved plans

{
  "success": true,
  "total_original_plans": 50,
  "circuits": 5,
  "mutations_tested": 2123,
  "malformed_inputs_tested": 300,
  "saved_width_agreement": true,
  "all_original_plans_valid_both_methods": true,
  "oracle_disagreements": 0,
  "malformed_input_failures": 0,
  "per_operator_cap": 12,
  "rng_seed": 20261011,
  "scope": "STATIC dependency feasibility; no dynamic semantics or routing",
  "warnings": []
}

Per-circuit/operator results:
           circuit            operator  evaluated  statically_valid  oracle_matches  all_mutations_preserve_width  all_mutations_preserve_partition  mean_available_unique_neighborhood  valid_rate
 alu-bdd_288.qasm      chain_transfer         72                 8              72                          True                              True        

In [5]:
import pandas as pd
width = pd.read_csv(OUT_DIR / "width_and_original_plan_audit.csv")
cases = pd.read_csv(OUT_DIR / "adversarial_mutation_cases.csv")
summary = pd.read_csv(OUT_DIR / "neighborhood_diagnostic_summary.csv")
reasons = pd.read_csv(OUT_DIR / "rejection_reasons.csv")
print("Stored width vs checker:", int((width.saved_width == width.checker_width).sum()), '/',len(width))
print("Stored width vs previous batch validation:",int((width.saved_width == width.previous_actual_width).sum()), '/',len(width))
print("Reference agreement:",int(cases.oracle_agrees.sum()),'/',len(cases))
print("Operator summary:")
display(summary)
print("Reasons:")
display(reasons)

Stored width vs checker: 50 / 50
Stored width vs previous batch validation: 50 / 50
Reference agreement: 2123 / 2123
Operator summary:


,circuit,operator,evaluated,statically_valid,oracle_matches,all_mutations_preserve_width,all_mutations_preserve_partition,mean_available_unique_neighborhood,valid_rate
0,alu-bdd_288.qasm,chain_transfer,72,8,72,True,True,34.000000,0.111111
1,alu-bdd_288.qasm,node_swap_reconnect,72,9,72,True,True,14.500000,0.125000
2,alu-bdd_288.qasm,segment_reverse,13,0,13,True,True,2.230769,0.000000
3,alu-bdd_288.qasm,suffix_swap,53,12,53,True,True,8.849057,0.226415
4,mini_alu_305.qasm,chain_transfer,192,7,192,True,True,94.000000,0.036458
5,mini_alu_305.qasm,node_swap_reconnect,192,13,192,True,True,35.250000,0.067708
6,mini_alu_305.qasm,segment_reverse,74,0,74,True,True,4.675676,0.000000
7,mini_alu_305.qasm,suffix_swap,192,17,192,True,True,25.375000,0.088542
8,rd73_140.qasm,chain_transfer,72,1,72,True,True,89.333333,0.013889
9,rd73_140.qasm,node_swap_reconnect,72,3,72,True,True,35.833333,0.041667


Reasons:


,circuit,operator,checker_reason,count
0,alu-bdd_288.qasm,chain_transfer,illegal_reuse_boundary,64
1,alu-bdd_288.qasm,chain_transfer,ok_static,8
2,alu-bdd_288.qasm,node_swap_reconnect,illegal_reuse_boundary,63
3,alu-bdd_288.qasm,node_swap_reconnect,ok_static,9
4,alu-bdd_288.qasm,segment_reverse,illegal_reuse_boundary,13
5,alu-bdd_288.qasm,suffix_swap,illegal_reuse_boundary,41
6,alu-bdd_288.qasm,suffix_swap,ok_static,12
7,mini_alu_305.qasm,chain_transfer,dependency_cycle,16
8,mini_alu_305.qasm,chain_transfer,illegal_reuse_boundary,169
9,mini_alu_305.qasm,chain_transfer,ok_static,7


## What the results mean

- A `success: true` report supports the checker on the tested static circuits and mutations **only**. A false positive/negative can still exist outside the sampled set.
- `valid_rate` and `mean_available_unique_neighborhood` are **diagnostics of mutation feasibility**, not evidence that an operator improves routed depth or SWAPs.
- Any `oracle_disagreements.csv` row must be investigated individually. Do not weaken the oracle/checker just to make counts agree.
- Next: an **end-to-end semantic correctness** suite on a limited set of concrete compiled circuits, then physical evaluation of the neighborhood.